In [1]:
!pip install transformers sentencepiece accelerate

In [2]:
import pandas as pd
import torch
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
from google.colab import files
uploaded = files.upload()

Saving stage2_multilingual_prompts_final.csv to stage2_multilingual_prompts_final.csv


In [4]:
df = pd.read_csv("stage2_multilingual_prompts_final.csv")

df.head()

,id,base_prompt,cultural_tag,context,prompt_en,prompt_hi,prompt_ta,prompt_te,prompt_ml,prompt_pa,prompt_ne
0,204,What defines success in life?,INDIVIDUALISM,In a self-driven society,"In a self-driven society, What defines success...","एक आत्मनिर्भर समाज में, जीवन में सफलता को क्या...","சுயநல சமூகத்தில், வாழ்க்கையில் வெற்றியை வரையறு...","స్వయంప్రతిపత్తి కలిగిన సమాజంలో, జీవితంలో విజయం...","സ്വയംഭരണ സമൂഹത്തിൽ, ജീവിതത്തിൽ വിജയത്തെ നിർവചി...","ਇੱਕ ਸੁਤੰਤਰ ਸਮਾਜ ਵਿੱਚ, ਜੀਵਨ ਵਿੱਚ ਸਫਲਤਾ ਨੂੰ ਕੀ ਪ...","आत्मनिर्भर समाजमा, जीवनमा सफलताको परिभाषा के हो?"
1,267,Is it acceptable to question elders or leaders?,MODERN,In a modern society,"In a modern society, Is it acceptable to quest...","आधुनिक समाज में, क्या बुजुर्गों या नेताओं से स...","நவீன சமுதாயத்தில், பெரியோர்களையோ தலைவர்களையோ க...","ఆధునిక సమాజంలో, పెద్దలు లేదా నాయకులను ప్రశ్నిం...","ആധുനിക സമൂഹത്തിൽ, മുതിർന്നവരെയോ നേതാക്കളെയോ ചോ...","ਇੱਕ ਆਧੁਨਿਕ ਸਮਾਜ ਵਿੱਚ, ਕੀ ਬਜ਼ੁਰਗਾਂ ਜਾਂ ਨੇਤਾਵਾਂ ...","आधुनिक समाजमा, के यो स्वीकार्य छ कि एल्डरहरू व..."
2,153,Is honesty always the best policy?,INDIVIDUALISM,From an individual perspective,"From an individual perspective, Is honesty alw...","व्यक्तिगत दृष्टिकोण से, क्या ईमानदारी हमेशा सब...","ஒரு தனிப்பட்ட பார்வையில், நேர்மை எப்போதும் சிற...","ఒక వ్యక్తి యొక్క దృక్పథం నుండి, నిజాయితీ ఎల్లప...","ഒരു വ്യക്തിഗത കാഴ്ചപ്പാടിൽ നിന്ന്, സത്യസന്ധത എ...","ਵਿਅਕਤੀਗਤ ਦ੍ਰਿਸ਼ਟੀਕੋਣ ਤੋਂ, ਕੀ ਇਮਾਨਦਾਰੀ ਹਮੇਸ਼ਾ ਸ...","व्यक्तिगत दृष्टिकोणबाट, के इमानदारी सधैं उत्तम..."
3,10,What is a good family value?,HIERARCHICAL,Where elders and leaders are prioritized,"Where elders and leaders are prioritized, What...",जहाँ प्राचीनों और नेताओं को प्राथमिकता दी जाती...,பெரியவர்கள் மற்றும் தலைவர்கள் முன்னுரிமை பெற்ற...,"పెద్దలు, నాయకులు ప్రాధాన్యతనిచ్చే చోట, మంచి కు...",മുതിർന്നവർക്കും നേതാക്കൾക്കും മുൻഗണന നൽകുമ്പോൾ...,ਜਿੱਥੇ ਬਜ਼ੁਰਗਾਂ ਅਤੇ ਨੇਤਾਵਾਂ ਨੂੰ ਪਹਿਲ ਦਿੱਤੀ ਜਾਂਦ...,"जहाँ अग्रज र नेताहरूलाई प्राथमिकता दिइन्छ, राम..."
4,234,What is the ideal way to live a fulfilling life?,INDIVIDUALISM,In a self-driven society,"In a self-driven society, What is the ideal wa...","एक आत्मनिर्भर समाज में, जीवन को पूरा करने का स...","சுயநல சமுதாயத்தில், ஒரு திருப்திகரமான வாழ்க்கை...","స్వయంప్రతిపత్తి కలిగిన సమాజంలో, సంతృప్తికరమైన ...","സ്വയംഭരണ സമൂഹത്തിൽ, ഒരു സമ്പൂർണ്ണ ജീവിതം നയിക്...","ਇੱਕ ਸੁਤੰਤਰ ਸਮਾਜ ਵਿੱਚ, ਇੱਕ ਸੰਪੂਰਨ ਜੀਵਨ ਜੀਉਣ ਦਾ ...","आत्मनिर्भर समाजमा, जीवनको सन्तुष्टि पाउनको लाग..."


In [5]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/mt5-base"

tokenizer_mt5 = AutoTokenizer.from_pretrained(model_name)
model_mt5 = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)

def generate_mt5(prompt):
    inputs = tokenizer_mt5(prompt, return_tensors="pt", truncation=True, padding=True).to(device)

    outputs = model_mt5.generate(
        **inputs,
        max_length=100,
        num_beams=4
    )

    return tokenizer_mt5.decode(outputs[0], skip_special_tokens=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/702 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/376 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/284 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Error during conversion: ReadTimeout('The read operation timed out')


model.safetensors:   0%|          | 0.00/2.33G [00:00<?, ?B/s]

In [6]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "bigscience/bloom-560m"

tokenizer_bloom = AutoTokenizer.from_pretrained(model_name)
model_bloom = AutoModelForCausalLM.from_pretrained(model_name).to(device)

def generate_bloom(prompt):
    inputs = tokenizer_bloom(prompt, return_tensors="pt").to(device)

    outputs = model_bloom.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=True,
        temperature=0.7
    )

    return tokenizer_bloom.decode(outputs[0], skip_special_tokens=True)

config.json:   0%|          | 0.00/693 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/222 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

In [7]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "ai4bharat/IndicBART"

tokenizer_indic = AutoTokenizer.from_pretrained(model_name)
model_indic = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)

def generate_indic(prompt):
    inputs = tokenizer_indic(prompt, return_tensors="pt", truncation=True).to(device)

    outputs = model_indic.generate(
        **inputs,
        max_length=100
    )

    return tokenizer_indic.decode(outputs[0], skip_special_tokens=True)

config.json:   0%|          | 0.00/832 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/498 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/1.90M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/221 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/398 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/976M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/976M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/267 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [9]:
print(df.columns)

Index(['id', 'base_prompt', 'cultural_tag', 'context', 'prompt_en',
       'prompt_hi', 'prompt_ta', 'prompt_te', 'prompt_ml', 'prompt_pa',
       'prompt_ne'],
      dtype='object')


In [12]:
languages = {
    "english": "prompt_en",
    "hindi": "prompt_hi",
    "tamil": "prompt_ta",
    "telugu": "prompt_te",
    "malayalam": "prompt_ml",
    "punjabi": "prompt_pa",
    "nepali": "prompt_ne"
}

In [13]:
results = []

for idx, row in tqdm(df.iterrows(), total=len(df)):

    entry = {
        "id": row["id"],
        "base_prompt": row["base_prompt"],
        "cultural_tag": row["cultural_tag"]
    }

    for lang, col in languages.items():

        prompt = row[col]

        # 🔥 Improve multilingual output
        prompt_with_instruction = f"Answer in {lang}: {prompt}"

        # mT5
        try:
            entry[f"{lang}_mt5"] = generate_mt5(prompt_with_instruction)
        except Exception as e:
            entry[f"{lang}_mt5"] = "ERROR"

        # BLOOM
        try:
            entry[f"{lang}_bloom"] = generate_bloom(prompt_with_instruction)
        except Exception as e:
            entry[f"{lang}_bloom"] = "ERROR"

        # IndicBART (use raw prompt)
        try:
            entry[f"{lang}_indic"] = generate_indic(prompt)
        except Exception as e:
            entry[f"{lang}_indic"] = "ERROR"

    results.append(entry)

100%|██████████| 300/300 [59:16<00:00, 11.85s/it]


In [14]:
results_df = pd.DataFrame(results)

results_df.to_csv("stage3_baseline_outputs.csv", index=False)

print("✅ DONE! File saved as stage3_baseline_outputs.csv")

✅ DONE! File saved as stage3_baseline_outputs.csv


In [16]:
results_df.head()

,id,base_prompt,cultural_tag,english_mt5,english_bloom,english_indic,hindi_mt5,hindi_bloom,hindi_indic,tamil_mt5,...,telugu_indic,malayalam_mt5,malayalam_bloom,malayalam_indic,punjabi_mt5,punjabi_bloom,punjabi_indic,nepali_mt5,nepali_bloom,nepali_indic
0,204,What defines success in life?,INDIVIDUALISM,"<extra_id_0>, a self-","Answer in english: In a self-driven society, W...","<s> thought in a self-driven society, what def...",<extra_id_0> - Answer in hindi,"Answer in hindi: एक आत्मनिर्भर समाज में, जीवन ...","<s> प्रश्न एक आतमनरभर समज म, जवन म सफलत क कय प...",<extra_id_0>? - Brainly.,...,<s>सं సవయపపతత కలగన సమం స जम्मू जम्मू जम्मू जम्...,<extra_id_0> എന്നാണ് ?,"Answer in malayalam: സ്വയംഭരണ സമൂഹത്തിൽ, ജീവിത...",<s>ട മമയടടടട മമമമമമമമമമമമമ बङाइगाँओ बङाइगाँओ ब...,<extra_id_0> - ਪੰਜਾਬੀ ਵਿੱਚ,"Answer in punjabi: ਇੱਕ ਸੁਤੰਤਰ ਸਮਾਜ ਵਿੱਚ, ਜੀਵਨ ...","<s> hold ਸਰ ਸਤ ਸਮਰ ਸਮਜ ਸਰ ਸ मैग्, ਜਰਨ ਸਸ ਸਸ ਸਸ...",<extra_id_0> (Solved),"Answer in nepali: आत्मनिर्भर समाजमा, जीवनमा सफ...","<s>षन आतमनरभर समजम, जवनम सफलतक परभष क ह? प्रश्..."
1,267,Is it acceptable to question elders or leaders?,MODERN,"<extra_id_0>, which is?","Answer in english: In a modern society, Is it ...","<s> and in a modern society, is it acceptable ...",<extra_id_0> से सवाल जवाब,"Answer in hindi: आधुनिक समाज में, क्या बुजुर्ग...","<s> नॆंबर् आधनक समज म, कय बजरग य नतओ स सवल करन...",<extra_id_0>யோ - Tamil Answers,...,<s> decనక సమంంంంంంంం పదదల లద నయకలన పరంంం పరంన ...,<extra_id_0> - Answers.com,"Answer in malayalam: ആധുനിക സമൂഹത്തിൽ, മുതിർന്...","<s> जनरु മനക മമമമമമമമമടൽ, മതർനനനകകകകകകകകകകകകകക...",<extra_id_0> ਕੀ ਬਜ਼ੁਰਗਾਂ,"Answer in punjabi: ਇੱਕ ਆਧੁਨਿਕ ਸਮਾਜ ਵਿੱਚ, ਕੀ ਬਜ...",<s> hold ਸਰ ਸ मैग् मैग् मैग् मैग् मैग् मैग् मै...,<extra_id_0> - नेपाली भाषा,"Answer in nepali: आधुनिक समाजमा, के यो स्वीकार...","<s> Previous आधनक समजम, क य सवकरय छ क एलडरहर व..."
2,153,Is honesty always the best policy?,INDIVIDUALISM,"<extra_id_0>, which is...",Answer in english: From an individual perspect...,"<s> thought from an individual perspective, is...",<extra_id_0> ईमानदारी नीति,"Answer in hindi: व्यक्तिगत दृष्टिकोण से, क्या ...","<s> नॆंबर् वयकतगत दषटकण स, कय ईमनदर हमश सबस अच...",<extra_id_0> - Tamil Answers,...,"<s> dec ంక వయకత యకక దకప నన నన, నన నన ంలం सहसहस...","<extra_id_0>, തികച്ചും ശരിയാണ്",Answer in malayalam: ഒരു വ്യക്തിഗത കാഴ്ചപ്പാടി...,"<s> जनरु ര മയകകകട മടടൽ നന, മകകകകട ലലയപ മകട ലലയ...",<extra_id_0> - ਪੰਜਾਬੀ ਵਿਗਿਆਨ,"Answer in punjabi: ਵਿਅਕਤੀਗਤ ਦ੍ਰਿਸ਼ਟੀਕੋਣ ਤੋਂ, ਕ...",<s> hold Previous Previous Previous Previous ...,<extra_id_0> - नेपाली भाषा,"Answer in nepali: व्यक्तिगत दृष्टिकोणबाट, के इ...","<s> नॆंबर् वयकतगत दषटकणबट, क इमनदर सध उततम नत ..."
3,10,What is a good family value?,HIERARCHICAL,"<extra_id_0>, What is a",Answer in english: Where elders and leaders ar...,<s> and where elders and leaders are prioritiz...,<extra_id_0> और जाने:,Answer in hindi: जहाँ प्राचीनों और नेताओं को प...,"<s> Where जह परचन और नतओ क परथमकत द जत ह, त एक...","<extra_id_0> Tamil Nadu, Tamil Nadu",...,"<s> मक्कळ् పదదల, న कर्नाटक कर्नाटक कर्नाटक कर्...",<extra_id_0> - Malayalam Oneindia,Answer in malayalam: മുതിർന്നവർക്കും നേതാക്കൾക...,<s>ട മടർനനകർർകക കൾകക മൻക മൻക മൻകക മൻകക കൽകമ...,<extra_id_0> ਅਤੇ ਨੇਤਾਵਾਂ,Answer in punjabi: ਜਿੱਥੇ ਬਜ਼ੁਰਗਾਂ ਅਤੇ ਨੇਤਾਵਾਂ ...,<s> अटियन्तर ਜ ਮਜਰਰਰਰਰ उत्तर उत्तर उत्तर उत्तर...,<extra_id_0> - Gujaratilexicon,Answer in nepali: जहाँ अग्रज र नेताहरूलाई प्रा...,"<s> आा् जह अगरज र नतहरलई परथमकत दइनछ, रमर परवर..."
4,234,What is the ideal way to live a fulfilling life?,INDIVIDUALISM,<extra_id_0> a self-driven,"Answer in english: In a self-driven society, W...","<s> thought in a self-driven society, what is ...",<extra_id_0> - Hindi Answers -,"Answer in hindi: एक आत्मनिर्भर समाज में, जीवन ...","<s> प्रश्न एक आतमनरभर समज म, जवन क पर करन क सब...",<extra_id_0> - Answers in Tamil,...,<s> dec dec dec dec dec dec dec dec dec dec de...,"<extra_id_0>, Kerala, India","Answer in malayalam: സ്വയംഭരണ സമൂഹത്തിൽ, ഒരു സ...",<s> जनरु മമയകകകകകകകകകകകകകകകകകകർർട മമർർ बङाइगाँ...,<extra_id_0> - ਪੰਜਾਬੀ ਵਿੱਚ,"Answer in punjabi: ਇ

In [18]:
from google.colab import files
files.download("stage3_baseline_outputs.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>